In [26]:
import numpy as np
import pandas as pd

In [27]:
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url,sep='\t',header=None,names=['label','message'])
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


#### Exploring Dataset

In [28]:
# Shape of the data
dim = df.shape
print(f"Rows: {dim[0]} , Cols: {dim[1]}")
print()

# What does a normal message look like?
print("Normal Message: ")
print(df[df['label'] == 'ham'].sample(3))  # Normal language used in these messages,refers to a person mostly.
print()

# What does a Spam message look like?
print("Spam message")
print(df[df['label'] == 'spam'].sample(3)) # Use of CAPS, special characters, Numbers, "claim","Win","free" are common in these messages.

Rows: 5572 , Cols: 2

Normal Message: 
     label                                            message
3191   ham  Hi neva worry bout da truth coz the truth will...
3593   ham                                  I anything lor...
27     ham  Did you catch the bus ? Are you frying an egg ...

Spam message
     label                                            message
3595  spam  Do you want a New Nokia 3510i Colour Phone Del...
3828  spam  Congratulations U can claim 2 VIP row A Ticket...
2402  spam  Babe: U want me dont u baby! Im nasty and have...


In [29]:
# Looking for length of mesagges
df['length'] = df['message'].apply(len)

print(df.groupby('label')['length'].describe())

        count        mean        std   min    25%    50%    75%    max
label                                                                 
ham    4825.0   71.482487  58.440652   2.0   33.0   52.0   93.0  910.0
spam    747.0  138.670683  28.873603  13.0  133.0  149.0  157.0  223.0


The mean length of spam messages are higher than normal messages

In [30]:
import nltk
nltk.download('stopwords') # We download the list of stopwords into our local machine from NLTK (Natural Language Toolkit) library. (we only run it once)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Stopwords are common words in a language (like "and", "the", "is", "in", etc.) that are often filtered out during natural language processing (NLP) tasks because they usually don’t add much meaningful information.

### Cleaning the data

In [31]:
import string 
from nltk.corpus import stopwords # to access the downloaded stopwords

# PREPROCESSING
def preprocessed_text(text):
    # Convert to lower case
    text = text.lower()

    # Remove punctuations
    text_no_punc = ''.join([char for char in text if char not in string.punctuation])

    # Remove stop words
    words = text_no_punc.split()
    word_no_stop = [word for word in words if word not in stopwords.words('english')]

    return ' '.join(word_no_stop)

df['cleaned_message'] = df['message'].apply(preprocessed_text)

print("Dataset after preprocessing")
print(df.head().to_string())

Dataset after preprocessing
  label                                                                                                                                                      message  length                                                                                                                          cleaned_message
0   ham                                              Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...     111                                                       go jurong point crazy available bugis n great world la e buffet cine got amore wat
1   ham                                                                                                                                Ok lar... Joking wif u oni...      29                                                                                                                  ok lar joking wif u oni
2  spam  Free entry in 2 a wkly comp to wi

### Feature Extraction (Vectorization)

In [32]:
from sklearn.feature_extraction.text import CountVectorizer  # Used to convert the message text into numbers (using bag of words model)
vectorization = CountVectorizer()

X = vectorization.fit_transform(df['cleaned_message'])
y = df['label']  # What we want to predict

# Shape of the vectors
print(f"X Shape: {X.shape}")
print(f"y Shape: {y.shape}")

X Shape: (5572, 9437)
y Shape: (5572,)


X Shape: (5572, 5392) Means Total 5572 messages in which 5392 unique words   
y Shape: (5572,) 5572 corresponding labels

### Splitting Data

In [33]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Shapes of Splitted data
print(f"X Train size: {X_train.shape}")
print(f"X Test size: {X_test.shape}")
print(f"Y Train size: {Y_train.shape}")
print(f"Y Test size: {Y_test.shape}")

X Train size: (4457, 9437)
X Test size: (1115, 9437)
Y Train size: (4457,)
Y Test size: (1115,)


### Training

In [34]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train,Y_train)


,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


### Testing

In [38]:
example_message = [
    "URGENT! You have won a 1 week FREE membership in our £1,000,000 Prize Jackpot! Txt the word: CLAIM to No: 81010",
    "Hey, are you coming to the game tonight?",
    "Congratulations! You've been selected to receive a free cruise. Text CRUISE to 55555 to claim.",
    "Hey i hope u r free, wanna have some fun!"
]

cleaned_example = [preprocessed_text(msg) for msg in example_message]
vectorized_example = vectorization.transform(cleaned_example)
prediction = model.predict(vectorized_example)

for msg, pred in zip(example_message,prediction):
    print(f"Message: {msg}\n Predicted: {pred.upper()}\n")

Message: URGENT! You have won a 1 week FREE membership in our £1,000,000 Prize Jackpot! Txt the word: CLAIM to No: 81010
 Predicted: SPAM

Message: Hey, are you coming to the game tonight?
 Predicted: HAM

Message: Congratulations! You've been selected to receive a free cruise. Text CRUISE to 55555 to claim.
 Predicted: SPAM

Message: Hey i hope u r free, wanna have some fun!
 Predicted: HAM



## Evaluation Metrics

In [40]:
from sklearn.metrics import accuracy_score,classification_report

pred = model.predict(X_test)
accuracy = accuracy_score(Y_test,pred)
print(f"Model accuracy: {accuracy*100:.2f}%")
print("Classification Report: ")
print(classification_report(Y_test,pred))

Model accuracy: 97.49%
Classification Report: 
              precision    recall  f1-score   support

         ham       0.99      0.98      0.99       966
        spam       0.88      0.95      0.91       149

    accuracy                           0.97      1115
   macro avg       0.93      0.96      0.95      1115
weighted avg       0.98      0.97      0.98      1115

